# E-commerce Data Engineering Lab

In this notebook I take a raw file of 500 online orders and walk it through the 12-step data engineering road-map: **ingest → wrangle → clean → transform → feature-engineer → aggregate → serialize**. It ends with one analytical insight.

**Data used**

| Role | File | Source |
|---|---|---|
| Primary transactions | `data/raw/transactions.csv` (500 rows) | Synthetic. It is generated by [`scripts/generate_transactions.py`](scripts/generate_transactions.py) with a fixed seed, and the dirty values are injected on purpose so there is real grime to find. |
| Secondary metadata | `data/raw/us_cities_top_1k.csv` (1,000 rows) | Plotly's open [US cities top 1k](https://github.com/plotly/datasets/blob/master/us-cities-top-1k.csv) dataset: city, state, population, latitude and longitude. |

I use the secondary file for two things: to **enrich** each order with its city's state and population, and as the second source for the **Data Dictionary** at the end.

## Setup

All file paths are defined once here. The notebook uses paths relative to the repository root, so it runs the same after a fresh `git clone`.

In [1]:
import json
import re
from collections import Counter, defaultdict, namedtuple
from dataclasses import asdict, astuple, dataclass, fields
from datetime import date, datetime
from pathlib import Path

import pandas as pd

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

RAW_TRANSACTIONS = RAW_DIR / "transactions.csv"
RAW_CITIES = RAW_DIR / "us_cities_top_1k.csv"
CLEAN_CSV = PROCESSED_DIR / "transactions_clean.csv"
CLEAN_JSON = PROCESSED_DIR / "transactions_clean.json"
REVENUE_JSON = PROCESSED_DIR / "revenue_by_city.json"

## Step 1 — Hello, Data!

I read every column **as text** (`dtype=str`) and turn off pandas' automatic missing-value detection (`keep_default_na=False`). I want to see the file exactly as it was written.

If I let pandas parse it, `"N/A"` and `"NULL"` would silently become `NaN` while `"none"` would stay a string. The grime would be half-hidden before I had even looked at it.

In [2]:
raw_df = pd.read_csv(RAW_TRANSACTIONS, dtype=str, keep_default_na=False)

print(f"Loaded {len(raw_df)} rows x {raw_df.shape[1]} columns")
raw_df.head(3)

Loaded 500 rows x 8 columns


,order_id,date,customer_id,product,price,quantity,coupon_code,shipping_city
0,ORD-10001,2025-01-01,C0076,USB-C Hub,39.99,2,,Phoenix
1,ORD-10002,01/02/2025,C0134,Mechanical Keyboard,89.99,3,,Atlanta
2,ORD-10003,2025-01-02,C0031,Wireless Mouse,24.99,2,WELCOME5,Denver


## Step 2 — Pick the Right Container

| Container | Where I use it | Why |
|---|---|---|
| **Class** (`@dataclass`) | One order: `Transaction` | An order must change during cleaning and carry behaviour (`clean()`, `total()`). A namedtuple is immutable, and a dict cannot hold methods. |
| **namedtuple** | `CityInfo`, `PriceProfile` | These are small, fixed, read-only records. A namedtuple gives named fields (`info.state`) with no extra code, and its immutability protects reference data. |
| **dict** | `city_lookup`: city name → `CityInfo` | Enrichment means "look up by key" thousands of times, and a dict does that in constant time. |
| **set** | Unique cities, unique customers | A set removes duplicates automatically, so `len(set(...))` is the unique count. |

## Step 3 — Implement Functions and Data Structure

**The functions.** Each helper does one conversion and returns `None` when a value cannot be used. The helpers never guess.

Each one also **returns non-string input unchanged**. That makes cleaning *idempotent*: running `clean()` on an already-clean record changes nothing. Step 7 relies on this to measure the raw and cleaned data with the same code.

In [3]:
DATE_FORMATS = ("%Y-%m-%d", "%m/%d/%Y")          # ISO first; US format seen in the raw file
COUPON_PLACEHOLDERS = {"", "N/A", "NA", "NONE", "NULL"}
MAX_QUANTITY = 50                                # nobody orders 999 monitors in one line
UNKNOWN_CUSTOMER = "UNKNOWN"


def normalize_city(value: str) -> str:
    """'  new york ' -> 'New York': trim, collapse inner spaces, title-case."""
    return " ".join(value.split()).title()


def parse_price(value: str | float | None) -> float | None:
    """'$24.99' -> 24.99. Returns None when the text is not a number."""
    if not isinstance(value, str):
        return value
    try:
        return float(value.replace("$", "").strip())
    except ValueError:
        return None


def parse_quantity(value: str | int | None) -> int | None:
    """'3' -> 3. Blank or non-numeric text -> None."""
    if not isinstance(value, str):
        return value
    text = value.strip()
    return int(text) if text.isdigit() else None


def parse_date(value: str | date | None) -> date | None:
    """Accepts '2025-03-14' or '03/14/2025'. Anything else -> None."""
    if not isinstance(value, str):
        return value
    for date_format in DATE_FORMATS:
        try:
            return datetime.strptime(value.strip(), date_format).date()
        except ValueError:
            continue
    return None


def normalize_coupon(value: str | None) -> str | None:
    """' save10' -> 'SAVE10'. Placeholders such as 'N/A' or 'none' -> None (no coupon)."""
    if not isinstance(value, str):
        return value
    code = value.strip().upper()
    return None if code in COUPON_PLACEHOLDERS else code


def coupon_discount(code: str | None) -> float:
    """Discount rate from the number inside the code: 'SAVE15' -> 0.15. No number -> 0.0."""
    if not isinstance(code, str):
        return 0.0
    match = re.search(r"\d+", code)
    return int(match.group()) / 100 if match else 0.0

**The data structure.** `Transaction` is one order line.

- It starts out holding the raw text.
- `clean()` returns a **new**, typed `Transaction` rather than changing itself, so the raw version stays available for before/after comparisons.
- `problems()` lists everything that makes the record unusable.
- `total()` is the net amount paid after the coupon.

`OrderBook` is a collection of transactions. It has its own batch-level `clean()`, which also removes duplicates, and a `total()` for revenue.

In [4]:
@dataclass
class Transaction:
    """One order line. Holds raw text until clean() returns a typed copy."""

    order_id: str
    date: str | date | None
    customer_id: str
    product: str
    price: str | float | None
    quantity: str | int | None
    coupon_code: str | None
    shipping_city: str

    @classmethod
    def from_row(cls, row: dict) -> "Transaction":
        """Build from one CSV row (a dict), keeping only the fields the class defines."""
        return cls(**{field.name: row[field.name] for field in fields(cls)})

    def clean(self) -> "Transaction":
        return Transaction(
            order_id=self.order_id.strip(),
            date=parse_date(self.date),
            customer_id=self.customer_id.strip() or UNKNOWN_CUSTOMER,
            product=self.product.strip(),
            price=parse_price(self.price),
            quantity=parse_quantity(self.quantity),
            coupon_code=normalize_coupon(self.coupon_code),
            shipping_city=normalize_city(self.shipping_city),
        )

    def problems(self) -> list[str]:
        """Reasons this (cleaned) record cannot be used; an empty list means it is valid."""
        issues = []
        if self.date is None:
            issues.append("unreadable date")
        if self.price is None or self.price <= 0:
            issues.append("missing or non-positive price")
        if self.quantity is None:
            issues.append("missing quantity")
        elif not 1 <= self.quantity <= MAX_QUANTITY:
            issues.append(f"quantity outside 1-{MAX_QUANTITY}")
        return issues

    def discount_rate(self) -> float:
        return coupon_discount(self.coupon_code)

    def total(self) -> float:
        """Net amount paid: price x quantity, minus the coupon discount."""
        return round(self.price * self.quantity * (1 - self.discount_rate()), 2)


class OrderBook:
    """A collection of transactions with batch-level cleaning and revenue."""

    def __init__(self, transactions: list[Transaction]) -> None:
        self.transactions = transactions

    def __len__(self) -> int:
        return len(self.transactions)

    def clean(self) -> tuple["OrderBook", dict]:
        """Clean every record, drop exact duplicates and invalid rows; return the result and a report."""
        cleaned = [t.clean() for t in self.transactions]

        # Deduplicate on the *cleaned* values, so the same order typed two ways still counts once.
        seen, unique = set(), []
        for t in cleaned:
            key = astuple(t)
            if key not in seen:
                seen.add(key)
                unique.append(t)

        valid = [t for t in unique if not t.problems()]
        reasons = Counter(issue for t in unique for issue in t.problems())

        report = {
            "rows before": len(cleaned),
            "exact duplicates removed": len(cleaned) - len(unique),
            "invalid rows removed": len(unique) - len(valid),
            "rows after": len(valid),
            **{f"  reason: {reason}": count for reason, count in reasons.items()},
        }
        return OrderBook(valid), report

    def total(self) -> float:
        return round(sum(t.total() for t in self.transactions), 2)

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([asdict(t) for t in self.transactions])

**Using it.** I populate a list with the first three rows, then clean each one. The raw prices are text. The cleaned prices are numbers, and `total()` applies the coupon: row 3 has `WELCOME5`, so it is 5% off.

In [5]:
sample = [Transaction.from_row(row) for row in raw_df.head(3).to_dict(orient="records")]

for raw in sample:
    clean = raw.clean()
    print(f"{raw.order_id}: raw price {raw.price!r:8} date {raw.date!r:13}"
          f"-> {clean.price} x {clean.quantity}, coupon {clean.coupon_code}, total {clean.total()}")

ORD-10001: raw price '39.99'  date '2025-01-01' -> 39.99 x 2, coupon None, total 79.98
ORD-10002: raw price '89.99'  date '01/02/2025' -> 89.99 x 3, coupon None, total 269.97
ORD-10003: raw price '24.99'  date '2025-01-02' -> 24.99 x 2, coupon WELCOME5, total 47.48


## Step 4 — Bulk Loaded

Now I load everything. `DataFrame.to_dict(orient="records")` turns each row into a dict, and each dict becomes a `Transaction` in an `OrderBook`.

The city file is mapped the same way into a **dict of lists**: `city → [CityInfo, ...]`. I use a list because a city name is not unique in the US. In this file, "Portland" is both Portland, Oregon and Portland, Maine. A plain `city → CityInfo` dict would silently keep whichever row came last.

In [6]:
CityInfo = namedtuple("CityInfo", ["state", "population", "latitude", "longitude"])

raw_book = OrderBook([Transaction.from_row(row) for row in raw_df.to_dict(orient="records")])

cities_df = pd.read_csv(RAW_CITIES)
city_lookup: dict[str, list[CityInfo]] = defaultdict(list)
for row in cities_df.to_dict(orient="records"):
    city_lookup[row["City"]].append(CityInfo(row["State"], row["Population"], row["lat"], row["lon"]))

print(f"OrderBook: {len(raw_book)} transactions")
print(f"city_lookup: {len(city_lookup)} distinct names covering {len(cities_df)} cities")
print("city_lookup['Portland'] =", city_lookup["Portland"])

OrderBook: 500 transactions
city_lookup: 925 distinct names covering 1000 cities
city_lookup['Portland'] = [CityInfo(state='Oregon', population=609456, latitude=45.523062200000005, longitude=-122.6764816), CityInfo(state='Maine', population=66318, latitude=43.661471, longitude=-70.2553259)]


## Step 5 — Quick Profiling

This is a first look at the numbers **before** any cleaning, taking the file at face value. `PriceProfile` is a namedtuple, so the result reads like a small report.

In [7]:
PriceProfile = namedtuple("PriceProfile", ["min", "mean", "max", "not_a_number"])


def profile_prices(raw_prices: pd.Series) -> PriceProfile:
    numbers = pd.to_numeric(raw_prices, errors="coerce")   # '$24.99' cannot be read -> NaN
    return PriceProfile(
        min=float(numbers.min()),
        mean=round(float(numbers.mean()), 2),
        max=float(numbers.max()),
        not_a_number=int(numbers.isna().sum()),
    )


price_profile = profile_prices(raw_df["price"])
raw_cities = set(raw_df["shipping_city"])
raw_customers = set(raw_df["customer_id"])

print(price_profile)
print(f"Unique city values:     {len(raw_cities)}")
print(f"  after normalize_city: {len({normalize_city(city) for city in raw_cities})}")
print(f"Unique customer values: {len(raw_customers)} (includes blank: {'' in raw_customers})")

PriceProfile(min=-249.99, mean=84.01, max=249.99, not_a_number=20)
Unique city values:     50
  after normalize_city: 15
Unique customer values: 140 (includes blank: True)


The profile already points at three problems:

- **The minimum price is negative.** A product cannot cost less than nothing.
- **Some prices are not numbers at all** (`not_a_number`), so the mean is being calculated on fewer rows than the file contains.
- **The city set is inflated.** The raw values contain many more "cities" than there really are. Once the spelling is normalised, they collapse to 15.

## Step 6 — Spot the Grime

I turn each suspicion into a check that counts the affected rows and shows one example. I use `repr()` for the examples so that stray spaces are visible.

In [8]:
def find_grime(df: pd.DataFrame) -> pd.DataFrame:
    """Count each kind of dirty value in the raw, all-text transactions."""
    quantity = pd.to_numeric(df["quantity"], errors="coerce")
    checks = {
        "exact duplicate row":               ("order_id", df.duplicated()),
        "price stored with a $ sign":        ("price", df["price"].str.startswith("$")),
        "negative price":                    ("price", df["price"].str.startswith("-")),
        "blank customer_id":                 ("customer_id", df["customer_id"].str.strip().eq("")),
        "blank quantity":                    ("quantity", df["quantity"].str.strip().eq("")),
        f"quantity above {MAX_QUANTITY}":    ("quantity", quantity.gt(MAX_QUANTITY)),
        "date not in YYYY-MM-DD format":     ("date", ~df["date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")),
        "city with odd case or spacing":     ("shipping_city", df["shipping_city"].ne(df["shipping_city"].map(normalize_city))),
        "coupon with odd case, spacing or placeholder":
            ("coupon_code", df["coupon_code"].ne(df["coupon_code"].map(normalize_coupon).fillna(""))),
    }
    rows = []
    for issue, (column, mask) in checks.items():
        example = repr(df.loc[mask, column].iloc[0]) if mask.any() else ""
        rows.append({"issue": issue, "column": column, "rows affected": int(mask.sum()), "example": example})
    return pd.DataFrame(rows)


grime = find_grime(raw_df)
grime

,issue,column,rows affected,example
0,exact duplicate row,order_id,10,'ORD-10008'
1,price stored with a $ sign,price,20,'$59.99'
2,negative price,price,6,'-249.99'
3,blank customer_id,customer_id,12,''
4,blank quantity,quantity,8,''
5,quantity above 50,quantity,3,'999'
6,date not in YYYY-MM-DD format,date,40,'01/02/2025'
7,city with odd case or spacing,shipping_city,60,'san diego'
8,"coupon with odd case, spacing or placeholder",coupon_code,40,'welcome5'


The secondary file has grime of its own that matters for the join: **ambiguous city names**.

In [9]:
ambiguous_names = {name for name, infos in city_lookup.items() if len(infos) > 1}
our_cities = {normalize_city(city) for city in raw_cities}

print(f"Names shared by more than one US city in the lookup: {len(ambiguous_names)}")
print(f"...of which appear in our orders: {sorted(our_cities & ambiguous_names)}")

Names shared by more than one US city in the lookup: 62
...of which appear in our orders: ['Portland']


**What each problem would break if I ignored it:**

1. **Duplicates** would double-count revenue for those orders.
2. **`$` prices** cannot be read as numbers. **Negative prices** would subtract revenue. Neither is a real sale price.
3. **Blank and 999 quantities** mean I cannot know what was actually sold.
4. **Mixed date formats**: `01/02/2025` is ambiguous in general (January 2 or February 1?), and it would break any date arithmetic.
5. **City and coupon spelling variants** split one real value into several groups, which corrupts any "per city" or "per coupon" result.
6. **Ambiguous city names** in the lookup would duplicate orders during the join (Step 8 shows this).

## Step 7 — Cleaning Rules

Every rule lives inside `Transaction.clean()` (per record) and `OrderBook.clean()` (per batch):

| Problem | Rule | Why |
|---|---|---|
| `$` prices | Strip the `$`, then convert to a number | The value is correct; only the formatting is wrong. |
| Negative prices | **Drop the row** | I cannot tell a refund from a typo, and guessing would invent revenue. |
| Blank quantity, or quantity above 50 | **Drop the row** | Without a real quantity there is no real revenue figure. |
| Blank `customer_id` | **Keep the row** and label the customer `UNKNOWN` | The sale still happened, so dropping it would understate revenue. |
| US-format dates | Parse both formats into one `date` type | Every date is readable once the format is known. |
| City spelling | Trim, collapse spaces, title-case | `"  new york"` and `"NEW YORK"` are the same city. |
| Coupon spelling and placeholders | Trim and upper-case. `N/A`, `none` and `NULL` become no coupon | They all mean the same thing. |
| Exact duplicates | Keep the first copy | This looks like a double-submitted order. |

In [10]:
clean_book, cleaning_report = raw_book.clean()
pd.Series(cleaning_report, name="count").to_frame()

,count
rows before,500
exact duplicates removed,10
invalid rows removed,16
rows after,474
reason: missing quantity,8
reason: missing or non-positive price,6
reason: quantity outside 1-50,3


The report counts reasons per row, and a row can fail for more than one reason. That is why the reasons can add up to more than the number of rows removed.

**Before vs after.** Because `clean()` is idempotent, the same function measures both books:

In [11]:
def quality_counts(book: OrderBook) -> dict[str, int]:
    records = book.transactions
    return {
        "rows": len(records),
        "duplicate order_ids": len(records) - len({t.order_id for t in records}),
        "distinct city spellings": len({t.shipping_city for t in records}),
        "distinct coupon values": len({t.coupon_code for t in records}),
        "blank customer_ids": sum(1 for t in records if not t.customer_id.strip()),
        "customers labelled UNKNOWN": sum(1 for t in records if t.customer_id == UNKNOWN_CUSTOMER),
        "rows that fail validation": sum(1 for t in records if t.clean().problems()),
    }


before_after = pd.DataFrame({"before": quality_counts(raw_book), "after": quality_counts(clean_book)})
before_after

,before,after
rows,500,474
duplicate order_ids,10,0
distinct city spellings,50,15
distinct coupon values,22,6
blank customer_ids,12,0
customers labelled UNKNOWN,0,12
rows that fail validation,16,0


## Step 8 — Transformations

### 8a. Coupon code → numeric discount

The rule is that **the number inside a code is its percentage off**: `SAVE15` means 15%, and `WELCOME5` means 5%. `FREESHIP` has no number, so it gives no price discount (it waives shipping, which this dataset does not record).

The table shows every raw spelling and what it becomes:

In [12]:
coupon_map = raw_df["coupon_code"].value_counts().rename_axis("raw value").reset_index(name="rows")
coupon_map["normalized"] = coupon_map["raw value"].map(normalize_coupon)
coupon_map["discount_rate"] = coupon_map["normalized"].map(coupon_discount)
coupon_map["raw value"] = coupon_map["raw value"].map(repr)      # make stray spaces visible
coupon_map

,raw value,rows,normalized,discount_rate
0,'',271,NaN,0.00
1,'SAVE10',57,SAVE10,0.10
2,'SAVE15',41,SAVE15,0.15
3,'WELCOME5',35,WELCOME5,0.05
4,'FREESHIP',29,FREESHIP,0.00
5,'SAVE20',27,SAVE20,0.20
6,'NULL',7,NaN,0.00
7,'none',6,NaN,0.00
8,'welcome5',4,WELCOME5,0.05
9,'SAVE10 ',3,SAVE10,0.10


### 8b. Objects → typed DataFrame

For column-wise work I convert the `OrderBook` back into a DataFrame. Each `Transaction` becomes a row through `dataclasses.asdict`. The date becomes a real `datetime64` column, and I add the discount rate from 8a.

In [13]:
orders = clean_book.to_dataframe()
orders["date"] = pd.to_datetime(orders["date"])
orders["discount_rate"] = [t.discount_rate() for t in clean_book.transactions]

orders.dtypes

order_id                   str
date             datetime64[s]
customer_id                str
product                    str
price                  float64
quantity                 int64
coupon_code                str
shipping_city              str
discount_rate          float64
dtype: object

### 8c. Enrichment: joining the city lookup

First, here is what a naive join by city name does:

In [14]:
naive_join = orders.merge(cities_df, left_on="shipping_city", right_on="City", how="left")
print(f"orders before join: {len(orders)}   after naive join: {len(naive_join)}")

orders before join: 474   after naive join: 506


The join **created extra rows**. Every Portland order matched both Portlands, so it was counted twice, and revenue would be inflated.

The fix is to resolve each city to exactly **one** match first:

- If the name is unique, use it.
- If it is ambiguous, take the most populous city with that name, and record in a `city_match` column that I made that choice.

Then I join with `validate="many_to_one"`, which makes pandas raise an error if any city still matches more than once.

In [15]:
def resolve_city(name: str) -> tuple[CityInfo | None, str]:
    matches = city_lookup.get(name, [])
    if not matches:
        return None, "not found"
    if len(matches) == 1:
        return matches[0], "exact"
    return max(matches, key=lambda info: info.population), "ambiguous: largest chosen"


city_rows = []
for city in sorted(set(orders["shipping_city"])):
    info, match = resolve_city(city)
    city_rows.append({"shipping_city": city, **(info._asdict() if info else {}), "city_match": match})

city_table = pd.DataFrame(city_rows).rename(columns={"population": "city_population"})
orders = orders.merge(city_table, on="shipping_city", how="left", validate="many_to_one")

print(f"orders after safe join: {len(orders)}")
city_table

orders after safe join: 474


,shipping_city,state,city_population,latitude,longitude,city_match
0,Atlanta,Georgia,447841,33.748995,-84.387982,exact
1,Austin,Texas,885400,30.267153,-97.743061,exact
2,Boston,Massachusetts,645966,42.360082,-71.058880,exact
3,Chicago,Illinois,2718782,41.878114,-87.629798,exact
4,Dallas,Texas,1257676,32.776664,-96.796988,exact
5,Denver,Colorado,649495,39.739236,-104.990251,exact
6,Houston,Texas,2195914,29.760427,-95.369803,exact
7,Los Angeles,California,3884307,34.052234,-118.243685,exact
8,Miami,Florida,417650,25.761680,-80.191790,exact
9,New York,New York,8405837,40.712784,-74.005941,exact


## Step 9 — Feature Engineering

| New column | How it is calculated |
|---|---|
| `gross_amount` | `price × quantity` |
| `order_total` | `gross_amount × (1 − discount_rate)`: the same formula as `Transaction.total()` |
| `discount_amount` | `gross_amount − order_total` |
| `days_since_purchase` | Days from the order date to the **snapshot date** |
| `order_month`, `order_weekday`, `is_weekend` | Calendar parts of the order date |

**Snapshot date.** I measure `days_since_purchase` from the day after the last order in the data, not from today. With `date.today()`, every re-run of this notebook would produce different numbers, so the instructor's run would not match mine.

In [16]:
SNAPSHOT_DATE = orders["date"].max() + pd.Timedelta(days=1)

orders["gross_amount"] = (orders["price"] * orders["quantity"]).round(2)
orders["order_total"] = (orders["gross_amount"] * (1 - orders["discount_rate"])).round(2)
orders["discount_amount"] = (orders["gross_amount"] - orders["order_total"]).round(2)
orders["days_since_purchase"] = (SNAPSHOT_DATE - orders["date"]).dt.days
orders["order_month"] = orders["date"].dt.strftime("%Y-%m")
orders["order_weekday"] = orders["date"].dt.day_name()
orders["is_weekend"] = orders["date"].dt.dayofweek >= 5

# The vectorised column and the class method must agree, or one of them is wrong.
assert abs(orders["order_total"].sum() - clean_book.total()) < 0.01

print(f"Snapshot date: {SNAPSHOT_DATE.date()}")
orders[["order_id", "date", "price", "quantity", "discount_rate", "gross_amount",
        "discount_amount", "order_total", "days_since_purchase", "order_weekday", "is_weekend"]].head()

Snapshot date: 2025-07-01


,order_id,date,price,quantity,discount_rate,gross_amount,discount_amount,order_total,days_since_purchase,order_weekday,is_weekend
0,ORD-10001,2025-01-01,39.99,2,0.00,79.98,0.0,79.98,181,Wednesday,False
1,ORD-10002,2025-01-02,89.99,3,0.00,269.97,0.0,269.97,180,Thursday,False
2,ORD-10003,2025-01-02,24.99,2,0.05,49.98,2.5,47.48,180,Thursday,False
3,ORD-10004,2025-01-02,59.99,1,0.15,59.99,9.0,50.99,180,Thursday,False
4,ORD-10005,2025-01-02,34.99,2,0.00,69.98,0.0,69.98,180,Thursday,False


## Step 10 — Mini-Aggregation

**Revenue per shipping city**, calculated two ways. The first uses a plain dict built from the `Transaction` objects. The second uses `pandas.groupby`, which also brings in the city population from the secondary source.

In [17]:
revenue_dict: dict[str, float] = defaultdict(float)
for t in clean_book.transactions:
    revenue_dict[t.shipping_city] += t.total()

revenue_by_city = (
    orders.groupby("shipping_city")
    .agg(orders=("order_id", "count"),
         revenue=("order_total", "sum"),
         city_population=("city_population", "first"))
    .sort_values("revenue", ascending=False)
)
revenue_by_city["avg_order_value"] = revenue_by_city["revenue"] / revenue_by_city["orders"]
revenue_by_city["revenue_per_100k_residents"] = (
    revenue_by_city["revenue"] / revenue_by_city["city_population"] * 100_000
)

assert all(abs(revenue_dict[city] - value) < 0.01 for city, value in revenue_by_city["revenue"].items())

print(f"Total revenue: ${clean_book.total():,.2f} from {len(clean_book)} orders")
revenue_by_city.round(2)

Total revenue: $69,721.76 from 474 orders


,orders,revenue,city_population,avg_order_value,revenue_per_100k_residents
shipping_city,,,,,
Boston,35,9905.50,645966,283.01,1533.44
New York,51,6743.60,8405837,132.23,80.23
Los Angeles,66,6529.88,3884307,98.94,168.11
Portland,32,6362.93,609456,198.84,1044.03
Austin,37,5644.39,885400,152.55,637.50
Philadelphia,28,5107.38,1553165,182.41,328.84
Seattle,32,4749.43,652405,148.42,727.99
Houston,33,4702.95,2195914,142.51,214.17
Dallas,27,3769.26,1257676,139.60,299.70


### Insight

**Order count is a poor stand-in for revenue.**

- **Los Angeles** has the most orders (66) but ranks only third in revenue, because its average order is just $99.
- **Boston** has about half as many orders (35) but the highest revenue ($9,905). 86% of its revenue comes from the two premium products (monitors and headphones), compared with 61% across all cities.
- **Per resident, the biggest markets are the least reached.** New York brings in about $80 per 100,000 residents, against Boston's $1,533.

**Two caveats:**

1. The data is synthetic, and each city has only 15–66 orders, so a handful of monitor orders can change a city's ranking.
2. Portland's per-resident figure depends on the assumption I made in Step 8. If these orders actually went to Portland, **Maine**, it would be about $9,595 per 100,000 residents instead of $1,044, roughly nine times higher. This is why I kept the `city_match` column.

## Step 11 — Serialization Checkpoint

I save the cleaned, enriched data in two formats:

- **CSV** for spreadsheets and quick inspection.
- **JSON** (`orient="records"`, one object per order) for web apps and APIs.

Dates are written as plain `YYYY-MM-DD` text so both formats agree. The per-city revenue dict is saved as its own small JSON file.

Then I **read both files back** and check that the row counts and revenue survived the round trip.

In [18]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

export = orders.assign(date=orders["date"].dt.strftime("%Y-%m-%d"))
export.to_csv(CLEAN_CSV, index=False)
export.to_json(CLEAN_JSON, orient="records", indent=2)
REVENUE_JSON.write_text(json.dumps({city: round(value, 2) for city, value in revenue_dict.items()}, indent=2))

round_trip = pd.DataFrame({
    "rows": [len(orders), len(pd.read_csv(CLEAN_CSV)), len(pd.read_json(CLEAN_JSON))],
    "revenue": [orders["order_total"].sum(),
                pd.read_csv(CLEAN_CSV)["order_total"].sum(),
                pd.read_json(CLEAN_JSON)["order_total"].sum()],
}, index=["in memory", CLEAN_CSV.name, CLEAN_JSON.name]).round(2)

for path in (CLEAN_CSV, CLEAN_JSON, REVENUE_JSON):
    print(f"{path}  ({path.stat().st_size / 1024:.1f} KB)")
round_trip

data/processed/transactions_clean.csv  (74.8 KB)
data/processed/transactions_clean.json  (264.9 KB)
data/processed/revenue_by_city.json  (0.3 KB)


,rows,revenue
in memory,474,69721.76
transactions_clean.csv,474,69721.76
transactions_clean.json,474,69721.76


In [19]:
print(json.dumps(json.loads(CLEAN_JSON.read_text())[0], indent=2))

{
  "order_id": "ORD-10001",
  "date": "2025-01-01",
  "customer_id": "C0076",
  "product": "USB-C Hub",
  "price": 39.99,
  "quantity": 2,
  "coupon_code": null,
  "shipping_city": "Phoenix",
  "discount_rate": 0.0,
  "state": "Arizona",
  "city_population": 1513367,
  "latitude": 33.4483771,
  "longitude": -112.0740373,
  "city_match": "exact",
  "gross_amount": 79.98,
  "order_total": 79.98,
  "discount_amount": 0.0,
  "days_since_purchase": 181,
  "order_month": "2025-01",
  "order_weekday": "Wednesday",
  "is_weekend": false
}
